**Step 1 & 2: Text data collection and preprocessing (Data Cleaning)**

In [1]:
import string
import os

def load_and_clean_captions(file_path):
    captions_dict = {}
    table = str.maketrans('', '', string.punctuation)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
                
            parts = line.split('\t') if '\t' in line else line.split(',', 1)
            
            if parts[0] == 'image' or len(parts) < 2:
                continue
                
            image_id, caption = parts[0], parts[1]
            
            image_id = image_id.split('#')[0].replace('.jpg', '')
            
            caption = caption.lower() 
            caption = caption.translate(table)
            
            words = [word for word in caption.split() if word.isalpha() and len(word) > 1]
            
            if image_id not in captions_dict:
                captions_dict[image_id] = []
            captions_dict[image_id].append(" ".join(words))
            
    return captions_dict

dataset_path = 'dataset/captions.txt' 
captions_mapping = load_and_clean_captions(dataset_path)

print(f"تعداد کل تصاویر پردازش شده: {len(captions_mapping)}")
print("-" * 30)

first_key = list(captions_mapping.keys())[0]
print(f"Image ID: {first_key}")
print(f"Captions: {captions_mapping[first_key]}")

تعداد کل تصاویر پردازش شده: 8091
------------------------------
Image ID: 1000268201_693b08cb0e
Captions: ['child in pink dress is climbing up set of stairs in an entry way', 'girl going into wooden building', 'little girl climbing into wooden playhouse', 'little girl climbing the stairs to her playhouse', 'little girl in pink dress going into wooden cabin']


**Step 3: Vocabulary Building and Word Filtering**

In [2]:
from collections import Counter

def build_vocabulary(captions_mapping, min_count=10):
    all_words = []
    for caps in captions_mapping.values():
        for cap in caps:
            all_words.extend(cap.split())
    
    word_counts = Counter(all_words)
    
    vocabulary = [word for word, count in word_counts.items() if count >= min_count]
    
    print(f"تعداد کل کلمات یکتا (قبل از فیلتر): {len(word_counts)}")
    print(f"تعداد کل کلمات لغت‌نامه نهایی (تکرار >= {min_count}): {len(vocabulary)}")
    
    return set(vocabulary)

vocab = build_vocabulary(captions_mapping, min_count=10)

تعداد کل کلمات یکتا (قبل از فیلتر): 8763
تعداد کل کلمات لغت‌نامه نهایی (تکرار >= 10): 1947


**Step 4: Data Splitting & Tokenization**

In [3]:
import random

all_img_ids = list(captions_mapping.keys())

random.seed(42) 
random.shuffle(all_img_ids)

split_index = int(len(all_img_ids) * 0.8)

train_ids = all_img_ids[:split_index]
test_ids = all_img_ids[split_index:]

train_captions = {}
for img_id in train_ids:
    if img_id in captions_mapping:
        train_captions[img_id] = []
        for cap in captions_mapping[img_id]:
            labeled_cap = 'startseq ' + cap + ' endseq'
            train_captions[img_id].append(labeled_cap)

print(f"تعداد کل تصاویر: {len(all_img_ids)}")
print(f"تعداد تصاویر آموزشی: {len(train_ids)}")
print(f"تعداد تصاویر تست: {len(test_ids)}")

sample_id = train_ids[0]
print(f"\nنمونه کپشن آموزشی برای {sample_id}:")
print(train_captions[sample_id][0])


تعداد کل تصاویر: 8091
تعداد تصاویر آموزشی: 6472
تعداد تصاویر تست: 1619

نمونه کپشن آموزشی برای 2874984466_1aafec2c9f:
startseq black and white dog is playing with sheep in field endseq


**Step 5 : Image feature extraction**

In [5]:
import os
import pickle
import numpy as np
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

all_img_ids = list(captions_mapping.keys()) 

project_dir = os.getcwd() 
images_dir = os.path.join(project_dir, 'dataset', 'Images')
weights_path = os.path.join(project_dir, 'models', 'inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5')
features_save_path = os.path.join(project_dir, 'features.pkl')

if not os.path.exists(weights_path):
    print(f" خطا: فایل وزن در این مسیر پیدا نشد: {weights_path}")
else:
    print(" در حال بارگذاری مدل InceptionV3...")
    base_model = InceptionV3(weights=weights_path, include_top=False, pooling='avg')
    print(" مدل با موفقیت بارگذاری شد.")

def extract_features(directory, img_ids):
    features = {}
    print(f" شروع استخراج ویژگی برای {len(img_ids)} تصویر...")
    
    for i, img_id in enumerate(img_ids):
        filename = os.path.join(directory, img_id + '.jpg')
        
        if not os.path.exists(filename): 
            print(f" هشدار: تصویر {filename} یافت نشد.")
            continue
            
        try:
            image = load_img(filename, target_size=(299, 299))
            image = img_to_array(image)
            image = np.expand_dims(image, axis=0)
            image = preprocess_input(image)
            
            feature = base_model.predict(image, verbose=0)
            features[img_id] = feature.reshape(-1)
            
            if (i+1) % 500 == 0: 
                print(f" پیشرفت: {i+1}/{len(img_ids)}")
        except Exception as e:
            print(f" خطا در پردازش تصویر {img_id}: {e}")
            
    return features

if os.path.exists(features_save_path):
    print(" فایل features.pkl موجود است. در حال بارگذاری...")
    with open(features_save_path, "rb") as f: 
        features = pickle.load(f)
    print(f" ویژگی‌ها بارگذاری شدند. تعداد: {len(features)}")
else:
    print("فایل ویژگی‌ها موجود نیست. در حال استخراج از تصاویر")
    features = extract_features(images_dir, all_img_ids) 
    with open(features_save_path, "wb") as f: 
        pickle.dump(features, f)
    print(" استخراج تمام شد و فایل features.pkl ذخیره گردید.")


 در حال بارگذاری مدل InceptionV3...
 مدل با موفقیت بارگذاری شد.
 فایل features.pkl موجود است. در حال بارگذاری...
 ویژگی‌ها بارگذاری شدند. تعداد: 8091


**Step 6: Text preparation and educational data production**

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import numpy as np

def to_lines(descriptions):
    all_desc = []
    for key in descriptions.keys():
        [all_desc.append(d) for d in descriptions[key]]
    return all_desc

def create_tokenizer(descriptions):
    lines = to_lines(descriptions)
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(lines)
    return tokenizer

def max_length(descriptions):
    lines = to_lines(descriptions)
    return max(len(d.split()) for d in lines)

tokenizer = create_tokenizer(train_captions)
vocab_size = len(tokenizer.word_index) + 1  
max_len = max_length(train_captions)

print(f' اندازه لغت‌نامه (Vocab Size): {vocab_size}')
print(f' حداکثر طول کپشن (Max Length): {max_len}')

def data_generator(descriptions, photos, tokenizer, max_length, vocab_size, batch_size):
    X1, X2, y = list(), list(), list()
    n = 0
    while True:
        for key, desc_list in descriptions.items():
            n += 1
            photo = photos[key][0] 
            for desc in desc_list:
                seq = tokenizer.texts_to_sequences([desc])[0]
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]
                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                    
                    X1.append(photo)
                    X2.append(in_seq)
                    y.append(out_seq)
            
            if n == batch_size:
                yield [np.array(X1), np.array(X2)], np.array(y)
                X1, X2, y = list(), list(), list()
                n = 0

 اندازه لغت‌نامه (Vocab Size): 7934
 حداکثر طول کپشن (Max Length): 34


**Step 7: Architecture and structure of the Image Captioning model**

In [7]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add

def define_model(vocab_size, max_length):
    inputs1 = Input(shape=(2048,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)

    inputs2 = Input(shape=(max_length,))
    se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
    se2 = Dropout(0.5)(se1)
    se3 = LSTM(256)(se2)

    decoder1 = add([fe2, se3])
    decoder2 = Dense(256, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)

    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    
    return model

model = define_model(vocab_size, max_len)

print(model.summary())

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)    │ (None, 34)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_2 (InputLayer)    │ (None, 2048)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 34, 256)           │       2,031,104 │ input_layer_3[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 2048)              │               0 │ input_layer_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 34, 256)           │               0 │ embedding[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal (NotEqual)          │ (None, 34)                │               0 │ input_layer_3[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 256)               │         524,544 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ (None, 256)               │         525,312 │ dropout_1[0][0],           │
│                               │                           │                 │ not_equal[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 256)               │               0 │ dense[0][0], lstm[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 256)               │          65,792 │ add[0][0]                  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 7934)              │       2,039,038 │ dense_1[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 5,185,790 (19.78 MB)

 Trainable params: 5,185,790 (19.78 MB)

 Non-trainable params: 0 (0.00 B)

None


**Step 8: Image caption generation model training**

In [8]:
import numpy as np
import tensorflow as tf

def get_batch(captions, features, tokenizer, max_len, vocab_size, batch_size):
    X1, X2, y = [], [], []
    n = 0

    for key, caption_list in captions.items():
        feature = features[key]

        for caption in caption_list:
            seq = tokenizer.texts_to_sequences([caption])[0]

            for i in range(1, len(seq)):
                in_seq, out_seq = seq[:i], seq[i]

                in_seq = tf.keras.preprocessing.sequence.pad_sequences(
                    [in_seq], maxlen=max_len
                )[0]

                out_seq = tf.keras.utils.to_categorical(
                    out_seq, num_classes=vocab_size
                )

                X1.append(feature)
                X2.append(in_seq)
                y.append(out_seq)

                n += 1

                if n == batch_size:
                    yield np.array(X1), np.array(X2), np.array(y)
                    X1, X2, y = [], [], []
                    n = 0

In [9]:
print(get_batch)

<function get_batch at 0x000001D350CD96C0>


In [10]:
batch_size = 64
epochs = 2
learning_rate = 0.001

In [11]:
loss_fn = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4) # مقدار استاندارد
print(" Loss function and Optimizer are ready.")

 Loss function and Optimizer are ready.


In [12]:
import tensorflow as tf
import numpy as np
import time
import math


best_loss = float('inf')

steps_per_epoch = math.ceil(len(train_captions) / batch_size)

print(" شروع ...")

for epoch in range(1, 3):
    print(f"\nEpoch {epoch}/3")
    start_time = time.time()

    batch_gen = get_batch(train_captions, features, tokenizer, max_len, vocab_size, batch_size)

    epoch_loss_avg = tf.keras.metrics.Mean()

    for step, (x1_batch, x2_batch, y_batch) in enumerate(batch_gen, start=1):
        with tf.GradientTape() as tape:
            preds = model([x1_batch, x2_batch])
            loss = loss_fn(y_batch, preds)

        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

        epoch_loss_avg.update_state(loss)

        if step % 50 == 0:
            print(f"Step {step} | Loss: {loss:.4f}", flush=True)

    epoch_time = time.time() - start_time
    current_loss = epoch_loss_avg.result()

    print(f"\n پایان اپوک {epoch} | میانگین Loss: {current_loss:.4f} | زمان: {epoch_time:.2f}s")

    if current_loss < best_loss:
        best_loss = current_loss
        model.save('models/best_model.h5')
        print(" مدل بهتر ذخیره شد")

print(" آموزش کامل شد")

 شروع ...

Epoch 1/3
Step 50 | Loss: 6.9259
Step 100 | Loss: 6.5637
Step 150 | Loss: 5.2634
Step 200 | Loss: 7.2899
Step 250 | Loss: 5.7399
Step 300 | Loss: 6.1418
Step 350 | Loss: 5.2640
Step 400 | Loss: 5.7460
Step 450 | Loss: 5.7643
Step 500 | Loss: 6.0797
Step 550 | Loss: 5.3480
Step 600 | Loss: 5.6393
Step 650 | Loss: 5.9958
Step 700 | Loss: 5.4725
Step 750 | Loss: 5.3608
Step 800 | Loss: 6.1417
Step 850 | Loss: 5.4722
Step 900 | Loss: 5.1550
Step 950 | Loss: 5.6474
Step 1000 | Loss: 5.8569
Step 1050 | Loss: 5.2040
Step 1100 | Loss: 5.4234
Step 1150 | Loss: 5.6430
Step 1200 | Loss: 5.2645
Step 1250 | Loss: 5.9612
Step 1300 | Loss: 5.8203
Step 1350 | Loss: 6.7330
Step 1400 | Loss: 5.8094
Step 1450 | Loss: 5.6828
Step 1500 | Loss: 6.6037
Step 1550 | Loss: 5.9092
Step 1600 | Loss: 5.1195
Step 1650 | Loss: 4.6278
Step 1700 | Loss: 5.4137
Step 1750 | Loss: 4.9815
Step 1800 | Loss: 4.7887
Step 1850 | Loss: 4.1297
Step 1900 | Loss: 4.4726
Step 1950 | Loss: 4.2563
Step 2000 | Loss: 5.6457


 پایان اپوک 1 | میانگین Loss: 5.1065 | زمان: 3682.34s
 مدل بهتر ذخیره شد

Epoch 2/3
Step 50 | Loss: 5.2989
Step 100 | Loss: 4.7098
Step 150 | Loss: 3.1516
Step 200 | Loss: 5.3674
Step 250 | Loss: 3.9338
Step 300 | Loss: 4.5538
Step 350 | Loss: 3.8866
Step 400 | Loss: 4.1805
Step 450 | Loss: 4.1546
Step 500 | Loss: 4.5957
Step 550 | Loss: 3.6924
Step 600 | Loss: 4.7256
Step 650 | Loss: 5.0208
Step 700 | Loss: 4.0741
Step 750 | Loss: 4.2009
Step 800 | Loss: 4.7174
Step 850 | Loss: 4.4801
Step 900 | Loss: 3.4782
Step 950 | Loss: 4.6112
Step 1000 | Loss: 4.4679
Step 1050 | Loss: 3.8313
Step 1100 | Loss: 4.1362
Step 1150 | Loss: 4.4910
Step 1200 | Loss: 4.2966
Step 1250 | Loss: 4.8809
Step 1300 | Loss: 4.9560
Step 1350 | Loss: 5.8534
Step 1400 | Loss: 5.0050
Step 1450 | Loss: 4.5024
Step 1500 | Loss: 5.4487
Step 1550 | Loss: 4.9731
Step 1600 | Loss: 4.0115
Step 1650 | Loss: 3.3137
Step 1700 | Loss: 4.2471
Step 1750 | Loss: 4.0119
Step 1800 | Loss: 3.6029
Step 1850 | Loss: 2.8299
Step 1900 


 پایان اپوک 2 | میانگین Loss: 4.1341 | زمان: 2899.95s
 مدل بهتر ذخیره شد
 آموزش کامل شد


**Step 9: Generate captions for new images**

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def idx_to_word(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None

def generate_caption(model, tokenizer, image, max_length):
    in_text = 'startseq'
    
    for i in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)
        
        yhat = model.predict([image, sequence], verbose=0)
        yhat = np.argmax(yhat)
        
        word = idx_to_word(yhat, tokenizer)
        
        if word is None:
            break
        
        in_text += ' ' + word
        
        if word == 'endseq':
            break
            
    return in_text

**Step 10: Model evaluation and caption generation for test images**

In [48]:
def generate_caption_final(model, tokenizer, photo, max_length):
    in_text = 'startseq'
    for i in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)
        photo_input = np.array(photo).reshape((1, 2048))
        
        yhat = model.predict([photo_input, sequence], verbose=0)
        yhat = np.argmax(yhat)
        
        word = tokenizer.index_word.get(yhat)
        if word is None or word == 'endseq':
            break
        in_text += ' ' + word
    return in_text.replace('startseq', '').strip()

import random
random_keys = random.sample(list(features.keys()), 5)

print("  : نمونه‌های تولید کپشن")
for key in random_keys:
    caption = generate_caption_final(model, tokenizer, features[key], max_len)
    print(f"تصویر: {key}")
    print(f"کپشن تولید شده: {caption}")
    print("-" * 50)


  : نمونه‌های تولید کپشن
تصویر: 3042484940_0975a5e486
کپشن تولید شده: man in red shirt and black shirt and black shirt and black shirt and black shirt and black shirt and black shirt and black shirt and black shirt and black shirt and black shirt
--------------------------------------------------
تصویر: 3461106572_920c8c0112
کپشن تولید شده: woman in black and woman and woman and woman and woman and woman and woman and woman and woman and woman and woman and woman and woman and woman and woman and woman and
--------------------------------------------------
تصویر: 3347701468_bb0001b035
کپشن تولید شده: football player in the ball
--------------------------------------------------
تصویر: 101669240_b2d3e7f17b
کپشن تولید شده: man in blue jacket is standing on the snow
--------------------------------------------------
تصویر: 3730011219_588cdc7972
کپشن تولید شده: man in blue shirt is jumping on the water
--------------------------------------------------
